Lấy tất cả bài viết trong ngày

In [2]:
import requests
from bs4 import BeautifulSoup
from datetime import date
import time
import random
import json
from langchain_text_splitters import RecursiveCharacterTextSplitter
from datetime import datetime, timedelta

# 1. Cấu hình

In [3]:

headers = {"User-Agent": "Mozilla/5.0"}

START_DATE = date(2026, 2, 8)
END_DATE   = date.today()

all_links = []


# 2. Lấy danh sách link

In [4]:

print(f"=== LẤY LINK {START_DATE.strftime('%d/%m/%Y')} → {END_DATE.strftime('%d/%m/%Y')} ===\n")

date_cursor = START_DATE
while date_cursor <= END_DATE:
    ngay = date_cursor.strftime("%Y-%m-%d")
    page = 1
    day_count = 0

    while True:
        if page == 1:
            url = f"https://dantri.com.vn/thoi-su/from/{ngay}/to/{ngay}.htm"
        else:
            url = f"https://dantri.com.vn/thoi-su/from/{ngay}/to/{ngay}/trang-{page}.htm"

        try:
            res = requests.get(url, headers=headers, timeout=10)
            soup = BeautifulSoup(res.text, "html.parser")
        except Exception as e:
            print(f"  ⚠️ {ngay} trang {page}: {e}")
            break

        articles = soup.find_all(
            "article",
            class_="article-item",
            attrs={"data-content-name": f"category-timeline_page_{page}"}
        )

        if not articles:
            break

        for art in articles:
            link = art.get("data-content-target")
            if link:
                if link.startswith("/"):
                    link = "https://dantri.com.vn" + link
                all_links.append(link)
                day_count += 1
        page += 1

    print(f"  {ngay}: {day_count} bài")
    date_cursor += timedelta(days=1)
    time.sleep(random.uniform(0.5, 1.0))

print(f"\n✅ TỔNG LINK: {len(all_links)}")


=== LẤY LINK 08/02/2026 → 01/04/2026 ===

  2026-02-08: 40 bài
  2026-02-09: 40 bài
  2026-02-10: 36 bài
  2026-02-11: 52 bài
  2026-02-12: 64 bài
  2026-02-13: 64 bài
  2026-02-14: 40 bài
  2026-02-15: 41 bài
  2026-02-16: 48 bài
  2026-02-17: 30 bài
  2026-02-18: 23 bài
  2026-02-19: 34 bài
  2026-02-20: 36 bài
  2026-02-21: 41 bài
  2026-02-22: 52 bài
  2026-02-23: 57 bài
  2026-02-24: 64 bài
  2026-02-25: 60 bài
  2026-02-26: 71 bài
  2026-02-27: 52 bài
  2026-02-28: 52 bài
  2026-03-01: 35 bài
  2026-03-02: 56 bài
  2026-03-03: 80 bài
  2026-03-04: 59 bài
  2026-03-05: 63 bài
  2026-03-06: 59 bài
  2026-03-07: 42 bài
  2026-03-08: 37 bài
  2026-03-09: 72 bài
  2026-03-10: 64 bài
  2026-03-11: 63 bài
  2026-03-12: 68 bài
  2026-03-13: 69 bài
  2026-03-14: 62 bài
  2026-03-15: 57 bài
  2026-03-16: 57 bài
  2026-03-17: 64 bài
  2026-03-18: 54 bài
  2026-03-19: 76 bài
  2026-03-20: 52 bài
  2026-03-21: 38 bài
  2026-03-22: 48 bài
  2026-03-23: 54 bài
  2026-03-24: 67 bài
  2026-03-25:

# 3. Hàm crawl

In [5]:
def crawl_article(url):
    res = requests.get(url, headers=headers, timeout=10)
    soup = BeautifulSoup(res.text, "html.parser")

    h1 = soup.find("h1")
    title = h1.get_text(strip=True) if h1 else ""

    time_tag = soup.find("time", attrs={"datetime": True})
    if time_tag:
        published_date = time_tag.get("datetime")
    else:
        published_date = ""

    paragraphs = []
    for p in soup.find_all("p"):
        text = p.get_text(strip=True)
        if text:
            paragraphs.append(text)

    body = "\n".join(paragraphs)

    return {
        "url": url,
        "title": title,
        "date": published_date,
        "content": body
    }

In [6]:
print("\n=== TEST 1 BÀI ===")
if all_links:
    test_url = all_links[0] # Lấy bài đầu tiên để test
    data = crawl_article(test_url)

    print("URL:", data["url"])
    print("TITLE:", data["title"])
    print("DATE:", data["date"])
    print("CONTENT:")
    print(data["content"][:500] + "...") # In 500 ký tự đầu
else:
    print("Không tìm thấy link nào để test.")



=== TEST 1 BÀI ===
URL: https://dantri.com.vn/thoi-su/khong-khi-lanh-lan-rong-vung-nui-cao-bac-bo-co-the-co-bang-gia-20260208192315407.htm
TITLE: Không khí lạnh lan rộng, vùng núi cao Bắc Bộ có thể có băng giá
DATE: 2026-02-09 00:00
CONTENT:
Tối 8/2, Trung tâm Dự báo khí tượng thủy văn quốc gia cho biết không khí lạnh đã ảnh hưởng đến Bắc Trung Bộ và hầu hết phía Tây Bắc Bộ. Nhiệt độ ở Bắc Bộ giảm 7-10 độ C, ở Bắc Trung Bộ giảm 1-3 độ C.
Dự báo đêm 8/2, không khí lạnh tiếp tục ảnh hưởng đến các nơi khác ở Bắc Bộ và Trung Bộ gây mưa và nhiệt độ nhiều nơi giảm sâu, trời rét đậm, rét hại dưới 3 độ C.
Khu vực Hà Nội đêm 8 và sáng 9/2 có mưa, trời rét đậm. Nhiệt độ thấp nhất ở thủ đô khoảng 10-13 độ C.
Cơ quan khí tượng nhận định từ đêm ...


# 4. Cấu hình Splitter

In [7]:

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", ".", "!", "?", ":", ";", ","]
)
print("Splitter sẵn sàng.")


Splitter sẵn sàng.


# 5. Xử lý hàng loạt

In [8]:
final_data = []
print("\n=== BẮT ĐẦU XỬ LÝ CHI TIẾT BÀI VIẾT ===")

for i, link in enumerate(all_links):
    print(f"[{i+1}/{len(all_links)}] Đang xử lý: {link}")

    article_data = crawl_article(link)

    if article_data and article_data["content"]:
        # Split text
        chunks = splitter.split_text(article_data["content"])

        # Lưu kết quả
        final_data.append({
            "url": link,
            "title": article_data["title"],
            "date": article_data["date"],
            "chunks": chunks
        })
        print(f"  -> OK: {len(chunks)} chunks | Date: {article_data['date']}")
    else:
        print("  -> Bỏ qua (không lấy được nội dung)")

    # Rate limiting
    sleep_time = random.uniform(1, 2)
    time.sleep(sleep_time)

print(f"\n=== HOÀN THÀNH ===")
print(f"Đã xử lý thành công: {len(final_data)} bài.")



=== BẮT ĐẦU XỬ LÝ CHI TIẾT BÀI VIẾT ===
[1/2786] Đang xử lý: https://dantri.com.vn/thoi-su/khong-khi-lanh-lan-rong-vung-nui-cao-bac-bo-co-the-co-bang-gia-20260208192315407.htm
  -> OK: 4 chunks | Date: 2026-02-09 00:00
[2/2786] Đang xử lý: https://dantri.com.vn/thoi-su/100-cu-tri-nhat-tri-gioi-thieu-ong-tran-sy-thanh-ung-cu-dbqh-khoa-moi-20260208225350822.htm
  -> OK: 3 chunks | Date: 2026-02-08 23:00
[3/2786] Đang xử lý: https://dantri.com.vn/thoi-su/hai-nguoi-di-xe-may-tu-vong-sau-khi-va-cham-voi-o-to-20260208224017447.htm
  -> OK: 2 chunks | Date: 2026-02-08 22:48
[4/2786] Đang xử lý: https://dantri.com.vn/thoi-su/ha-noi-dieu-chinh-thoi-gian-gio-cao-diem-va-bieu-do-chay-tau-metro-moi-20260208223640151.htm
  -> OK: 2 chunks | Date: 2026-02-08 22:44
[5/2786] Đang xử lý: https://dantri.com.vn/thoi-su/chu-tich-nuoc-luong-cuong-khat-vong-viet-nam-chinh-la-linh-hon-cua-dan-toc-20260208214247443.htm
  -> OK: 6 chunks | Date: 2026-02-08 21:52
[6/2786] Đang xử lý: https://dantri.com.vn/thoi

In [9]:

# 6. Lưu file JSON
output_file = "dulieu.json"
result = {
    "start_date": START_DATE.strftime("%Y-%m-%d"),
    "end_date":   END_DATE.strftime("%Y-%m-%d"),
    "total_articles": len(final_data),
    "articles": final_data
}
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=4)
print(f"✅ Đã lưu {len(final_data)} bài vào: {output_file}")


✅ Đã lưu 2783 bài vào: dulieu.json
